# Data Structuring & Combining

## Learning goals

- Decide whether to **stack** compatible datasets or **match** related tables.
- Use `pd.concat()` and `pd.merge()` with appropriate parameters.
- Distinguish inner, left, right and outer merges.
- Verify row counts, keys and missing values after combining.
- Build summary tables with `pivot_table()`.
- Recognize wide and long formats and use `melt()`.

**Guiding question:** what structure does the analysis need?

## 1. Why combine or restructure data?
Real data is often distributed across several tables. A logistics company might have monthly delivery files plus separate driver information.

Sometimes we need to **add more observations**. Sometimes we need to **add information about the same observations**. These are different problems.

In [ ]:
import pandas as pd

january = pd.DataFrame({
    "delivery_id": [101, 102, 103],
    "driver_id": ["D01", "D02", "D03"],
    "zone": ["North", "South", "North"],
    "distance_km": [12, 8, 15]
})
february = pd.DataFrame({
    "delivery_id": [104, 105, 106],
    "driver_id": ["D02", "D04", "D01"],
    "zone": ["East", "South", "East"],
    "distance_km": [10, 18, 7]
})
drivers = pd.DataFrame({
    "driver_id": ["D01", "D02", "D03", "D04"],
    "driver_name": ["Ana", "Bruno", "Carla", "David"],
    "vehicle": ["Van", "Bike", "Van", "Truck"]
})

display(january)
display(february)
display(drivers)

## 2. `concat()` — stacking compatible datasets

January and February contain the **same variables** but different deliveries. We want to stack February **under** January.

**But there is a problem:** look at the two original DataFrames. Neither has a `month` column. If we concatenate them as they are, the resulting table will not tell us which deliveries happened in January and which in February.

**Before combining the data**, let's add that missing context explicitly to each DataFrame. This way, we can later analyze deliveries by month (for example, calculate total distance travelled each month).

In [ ]:
# Preserve the source month before stacking the monthly files
january["month"] = "January"
february["month"] = "February"

display(january)
display(february)

In [ ]:
deliveries = pd.concat([january, february], axis=0, ignore_index=True)
display(deliveries)

### Understanding `axis`
- `axis=0` → stack vertically → add rows.
- `axis=1` → place horizontally → add columns.

For monthly files with the same variables, `axis=0` is normally appropriate. `ignore_index=True` creates a new sequential row index.

**Note:** `axis=1` aligns columns using index labels. To connect related business records by a key, prefer `merge()`. 

### 💡 Quick Check 1 — Preserve context before combining

A new DataFrame contains March deliveries but no `month` column. **Add the month first**, then concatenate it with the existing deliveries. Check that all three months can be identified in the result.

In [ ]:
march = pd.DataFrame({
    "delivery_id": [107, 108],
    "driver_id": ["D03", "D04"],
    "zone": ["North", "South"],
    "distance_km": [11, 14]
})

march["month"] = ________

all_deliveries = pd.________(
    [deliveries, ________],
    axis=_____,
    ignore_index=True
)
display(all_deliveries)

## 3. `merge()` — matching related information
The deliveries contain `driver_id`, while `drivers` contains names and vehicles. Both tables share the key `driver_id`, so we can **match related rows**.

In [ ]:
delivery_details = pd.merge(deliveries, drivers, on="driver_id")
display(delivery_details)

By default, `merge()` performs an **inner merge**: only keys found in both DataFrames are kept.

```text
concat → stack compatible tables
merge  → match related tables using a key
```

## 4. Choosing the merge type
Not every key necessarily appears in both DataFrames.

In [ ]:
routes = pd.DataFrame({
    "route_id": ["R1", "R2", "R3"],
    "zone": ["North", "South", "East"]
})
incidents = pd.DataFrame({
    "route_id": ["R2", "R3", "R4"],
    "incidents": [2, 1, 3]
})
display(routes)
display(incidents)

The `how` parameter determines which keys are kept.

| `how` | Keeps |
|---|---|
| `"inner"` | only keys present in both tables |
| `"left"` | all keys from the left table |
| `"right"` | all keys from the right table |
| `"outer"` | all keys from either table |

> **Which observations do I need to preserve?**

In [ ]:
print("INNER")
display(pd.merge(routes, incidents, on="route_id", how="inner"))

print("LEFT")
display(pd.merge(routes, incidents, on="route_id", how="left"))

print("RIGHT")
display(pd.merge(routes, incidents, on="route_id", how="right"))

print("OUTER")
display(pd.merge(routes, incidents, on="route_id", how="outer"))

### Different key names
If matching columns contain the same information but have different names, use `left_on` and `right_on`.

In [ ]:
zones = pd.DataFrame({
    "zone_code": ["North", "South", "East"],
    "manager": ["Laura", "Miguel", "Sara"]
})

deliveries_with_manager = pd.merge(
    deliveries, zones,
    left_on="zone", right_on="zone_code",
    how="left"
)
display(deliveries_with_manager)

### 💡 Quick Check 2 — Choose a merge type

Keep **all routes**, including routes without recorded incidents. Add incident information when it exists. Which `how` option is appropriate?

In [ ]:
routes_with_incidents = pd.merge(
    routes,
    incidents,
    on="route_id",
    how=________
)
display(routes_with_incidents)

## 5. Verify after combining data
A merge can run successfully and still produce an unexpected result. Check dimensions, missing values, keys and columns.

### Key relationships matter

One driver can appear in many deliveries, but each `driver_id` should appear only once in the driver lookup table. A repeated key on both sides may multiply rows during a merge.

```python
print(drivers["driver_id"].is_unique)
print(drivers["driver_id"].duplicated().sum())
```

**Optional safety check:** `pd.merge(deliveries, drivers, on="driver_id", validate="many_to_one")`.

In [ ]:
print("Before merge:", deliveries.shape)
print("After merge:", delivery_details.shape)

print("\nMissing values:")
display(delivery_details.isna().sum())

print("\nColumns:")
print(delivery_details.columns.tolist())

### 💡 Quick Check 4 — Complete the code
Verify the shape and missing values of `routes_with_incidents`.

In [ ]:
print(routes_with_incidents._____)
display(routes_with_incidents._____()._____())

## 6. Structuring data for a question
Sometimes all the information is already in one DataFrame, but its layout does not make the question easy to answer.

Suppose the operations manager wants to compare total delivery distance by **zone** and **vehicle**.

In [ ]:
display(delivery_details)

A row-level table is useful for storing observations. A summary table may be more useful for answering a specific question.

## 7. `pivot_table()` — building a summary table
A pivot table reorganizes data **and** performs an aggregation. We will explore aggregation in more detail in a separate lesson.

- `index` → rows
- `columns` → columns
- `values` → measure
- `aggfunc` → summary operation

In [ ]:
distance_summary = delivery_details.pivot_table(
    index="zone",
    columns="vehicle",
    values="distance_km",
    aggfunc="sum"
)
display(distance_summary)

This answers: **How much total delivery distance was covered in each zone by each vehicle type?**

```text
Rows?      → zone
Columns?   → vehicle
Measure?   → distance_km
Summary?   → sum
```

### 💡 Quick Check 5 — Complete the code
Create a pivot table showing the **average** delivery distance for each driver and zone.

In [ ]:
average_distance = delivery_details.________(
    index=________,
    columns=________,
    values=________,
    aggfunc=________
)
display(average_distance)

## 8. Wide and long data
The same information can be organized in different shapes.

### Wide format
Different repeated measurements occupy different columns.

In [ ]:
monthly_distance = pd.DataFrame({
    "driver": ["Ana", "Bruno", "Carla"],
    "jan_km": [120, 95, 110],
    "feb_km": [135, 105, 100],
    "mar_km": [128, 115, 125]
})
display(monthly_distance)

### Long format
The month can become a variable of its own:

```text
driver   month   distance_km
Ana      jan     120
Ana      feb     135
Ana      mar     128
```

Long format is often convenient because each column represents a variable.

## 9. `melt()` — wide to long
`melt()` transforms repeated-measurement columns into rows.

- `id_vars` → identifier columns
- `value_vars` → columns to reshape
- `var_name` → new variable column
- `value_name` → new values column

In [ ]:
distance_long = pd.melt(
    monthly_distance,
    id_vars=["driver"],
    value_vars=["jan_km", "feb_km", "mar_km"],
    var_name="month",
    value_name="distance_km"
)
display(distance_long)

### 💡 Quick Check 6 — Complete the code
Transform the wide table into long format. Keep `warehouse` as identifier and create `quarter` and `orders` columns.

In [ ]:
quarterly_orders = pd.DataFrame({
    "warehouse": ["Madrid", "Barcelona"],
    "q1": [320, 280],
    "q2": [350, 310]
})

orders_long = pd.________(
    quarterly_orders,
    id_vars=[________],
    value_vars=[________, ________],
    var_name=________,
    value_name=________
)
display(orders_long)

## 10. Choosing the right transformation

- Compatible tables with the same variables? → `pd.concat(...)`
- Related tables sharing a key? → `pd.merge(...)`
- Need an aggregated cross-tabular summary? → `df.pivot_table(...)`
- Repeated measurements in columns need to become rows? → `pd.melt(...)`

> **Restructure data when a different shape makes the analysis easier.**

## Final Challenge — Preparing a Delivery Report

**Estimated time: 8–10 minutes**

Your logistics company wants to prepare a report about its deliveries.

You already have the DataFrames created during this lesson.

**Your tasks:**

1. Combine the delivery records from January and February into a single DataFrame. Make sure you preserve the month of each delivery.
2. Add the information about each driver using the appropriate matching key.
3. Check that the resulting DataFrame contains the expected number of deliveries and that no unexpected rows were created.
4. Display the delivery ID, month, and driver information for the first five deliveries.

**Think about:**

- Why do we use `concat()` for the monthly deliveries but `merge()` for the driver information?
- What could happen if the matching key is not unique in the driver table?

**Bonus (optional):** Create a pivot table showing the number of deliveries per month and vehicle type.

In [ ]:
# Final Challenge — your solution
# Use january, february and drivers from the lesson.



## Summary

- **Stack** compatible observations with `pd.concat()`.
- **Match** related information using keys with `pd.merge()`.
- **Summarize** a cross-tabulation with `pivot_table()`.
- **Reshape** wide observations into long format with `melt()`.

After any combination, check row counts, keys and missing values.

### Optional practice

Using the delivery data, build a pivot table of **mean distance** by zone and vehicle. Explain what each cell means.